# 04 — Bronze: UN Comtrade bilateral trade totals

**Architecture:** Databricks Free Edition serverless cannot reach `comtradeapi.un.org` (DNS blocked). Data is extracted locally on Mac and uploaded to a Databricks Volume before this notebook runs.

## Steps before running this notebook

1. **Extract locally on Mac** (one-time per refresh):
   ```bash
   export COMTRADE_API_KEY="your-key"   # from password manager — never commit
   pip install -r extraction/requirements.txt
   python3 extraction/extract/comtrade_totals.py \
     --all-cemac-ecowas \
     --start-year 2010 \
     --end-year 2023 \
     --out data/raw/comtrade/cemac_ecowas_totals_2010_2023.jsonl
   ```
   Takes ~10–15 minutes (21 reporters × 14 years × 2 s/request).

2. **Upload JSONL to Volume** (replace `<profile>` with your Databricks profile name):
   ```bash
   databricks fs cp \
     data/raw/comtrade/cemac_ecowas_totals_2010_2023.jsonl \
     dbfs:/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/comtrade/cemac_ecowas_totals_2010_2023.jsonl \
     --profile <profile>
   ```

3. **Run this notebook top to bottom.** It reads the JSONL from the Volume, flattens observations, and writes `bronze.comtrade_raw`.


In [ ]:
# Cell 1 — Configuration
spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("USE SCHEMA bronze")

VOLUME_PATH = "/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/comtrade/"

CEMAC = ["120", "140", "148", "178", "226", "266"]
ECOWAS = ["204", "854", "132", "384", "270", "288", "324",
          "624", "430", "466", "562", "566", "686", "694", "768"]
ALL_REPORTERS = CEMAC + ECOWAS

YEARS = list(range(2010, 2024))

print(f"Catalog:   cemac_ecowas_aes_trade")
print(f"Schema:    bronze")
print(f"Volume:    {VOLUME_PATH}")
print(f"Reporters: {len(ALL_REPORTERS)} countries (M49 codes)")
print(f"Years:     {YEARS[0]}–{YEARS[-1]} ({len(YEARS)} years)")


In [ ]:
# Cell 2 — List JSONL files in the Volume
files = [f for f in dbutils.fs.ls(VOLUME_PATH) if f.name.endswith(".jsonl")]

if not files:
    raise FileNotFoundError(
        f"No .jsonl files found at {VOLUME_PATH}. "
        "Run comtrade_totals.py locally and upload the output first."
    )

print(f"Found {len(files)} JSONL file(s):")
for f in files:
    print(f"  {f.name}  ({f.size:,} bytes)")


In [ ]:
# Cell 3 — Load observations from JSONL envelopes
import json
from datetime import datetime, timezone

all_observations = []
loaded_at = datetime.now(timezone.utc)

for file_info in files:
    rdd = sc.textFile(file_info.path)
    for line in rdd.collect():
        line = line.strip()
        if not line:
            continue
        envelope = json.loads(line)
        for obs in envelope.get("payload", []):
            all_observations.append(obs)

print(f"Total raw observations (including World aggregates): {len(all_observations):,}")


In [ ]:
# Cell 4 — Inspect a sample observation (confirm field names before building the DataFrame)
if all_observations:
    sample = all_observations[0]
    print("Sample observation (first row):")
    for key, value in sample.items():
        print(f"  {key}: {value}")
    print()
    print(f"Total fields per observation: {len(sample)}")
else:
    print("No observations loaded. Check Cell 3 output.")


In [ ]:
# Cell 5 — Build Spark DataFrame (filter out World aggregate rows)
from pyspark.sql import Row

rows = []
for obs in all_observations:
    # partnerCode == 0 is the "World" aggregate — skip it to avoid double-counting
    if str(obs.get("partnerCode", "")) == "0":
        continue

    rows.append(Row(
        reporter_code=str(obs.get("reporterCode", "")),
        reporter_iso3=obs.get("reporterISO"),
        reporter_name=obs.get("reporterDesc"),
        partner_code=str(obs.get("partnerCode", "")),
        partner_iso3=obs.get("partnerISO"),
        partner_name=obs.get("partnerDesc"),
        year=int(obs.get("refYear", 0)),
        flow_code=obs.get("flowCode"),
        flow_desc=obs.get("flowDesc"),
        cmd_code=obs.get("cmdCode", "TOTAL"),
        hs_revision=obs.get("classificationCode"),
        trade_value_usd=float(obs["primaryValue"]) if obs.get("primaryValue") is not None else None,
        net_weight_kg=float(obs["netWgt"]) if obs.get("netWgt") is not None else None,
        source="un_comtrade",
        loaded_at=loaded_at,
    ))

df = spark.createDataFrame(rows)
print(f"DataFrame created: {df.count():,} rows (World aggregate rows removed)")
print()
df.printSchema()
print()
df.show(5, truncate=False)


In [ ]:
# Cell 6 — Write to bronze.comtrade_raw
spark.sql("DROP TABLE IF EXISTS bronze.comtrade_raw")

(df.write
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("bronze.comtrade_raw"))

print("Write complete.\n")


In [ ]:
# Cell 7 — Verify coverage and run a sanity check
print("Overall coverage:")
spark.sql("""
    SELECT
        COUNT(*)                                    AS total_rows,
        COUNT(DISTINCT reporter_iso3)               AS reporters,
        COUNT(DISTINCT partner_iso3)                AS partners,
        COUNT(DISTINCT year)                        AS years,
        MIN(year)                                   AS earliest_year,
        MAX(year)                                   AS latest_year,
        ROUND(SUM(trade_value_usd) / 1e12, 2)      AS total_trade_trillions_usd
    FROM bronze.comtrade_raw
""").show()

print("Per-reporter row count (reporters with gaps are expected for fragile states):")
spark.sql("""
    SELECT
        reporter_iso3,
        reporter_name,
        COUNT(*)                        AS rows,
        COUNT(DISTINCT year)            AS years_present,
        COUNT(DISTINCT partner_iso3)    AS distinct_partners
    FROM bronze.comtrade_raw
    GROUP BY reporter_iso3, reporter_name
    ORDER BY rows DESC
""").show(25, truncate=False)

print("Cameroon's top 10 export destinations in 2023:")
spark.sql("""
    SELECT
        partner_name,
        ROUND(trade_value_usd / 1e9, 2) AS exports_billions_usd
    FROM bronze.comtrade_raw
    WHERE reporter_iso3 = 'CMR'
      AND flow_code    = 'X'
      AND year         = 2023
    ORDER BY trade_value_usd DESC
    LIMIT 10
""").show(truncate=False)
